In [3]:
"""
Improved MCI-to-AD Conversion Prediction
Generates latent features for JMBayes2 joint modeling in R

ALL BUGS FIXED IN THIS VERSION:
Fix A — fusion_dim: (img_out_dim+1)*(tab_out_dim+1)*17, not *16.
         time_encoder outputs 16-dim; TensorFusion appends a 1 → 17-dim.
         *16 caused RuntimeError: mat1(16x283985) vs mat2(267280x128).
Fix B — BiLSTM h_final: torch.cat([h_n[-2], h_n[-1]]), not h_n[-1].
         h_n[-1] is only the last layer's FORWARD state (128-dim).
         temporal_proj expects 256-dim (lstm_hidden*2). This caused
         RuntimeError: mat1(16x128) vs mat2(256x128).
Fix C — forward() returns mmse_pred (singular) to match train_epoch.
         The variable was named mmse_preds inside forward() but the
         caller unpacks it as mmse_pred — kept consistent throughout.
Fix D — train_epoch loss loop iterates over mmse_pred.shape[1], which
         is now correct since forward() returns the right variable.
Fix E — L1 survival penalty re-enabled (was commented out).
Fix F — latent_dim=128 in CONFIG (was accidentally 64).
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import numpy as np
import os
import pickle
from lifelines.utils import concordance_index
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================
CONFIG = {
    'latent_dim':         128,   # Fix F: was 64 in the broken version
    'img_out_dim':        256,
    'tab_out_dim':        64,
    'lstm_hidden':        128,
    'lstm_layers':        2,
    'dropout':            0.4,
    'lr':                 3e-4,
    'weight_decay':       1e-3,
    'epochs':             100,
    'batch_size':         16,
    'accumulation_steps': 2,
    'max_seq_len':        10,
    'alpha_survival':     0.85,
    'alpha_mmse':         0.15,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}

# ============================================================================
# FEATURE DEFINITIONS
# ============================================================================
STATIC_FEATURES = [
    'age_bl', 'PTGENDER_encoded', 'PTEDUCAT', 'PTETHCAT_encoded',
    'PTRACCAT_encoded', 'PTMARRY_encoded'
]
TEMPORAL_FEATURES = [
    'time_from_baseline', 'AGE', 'age_since_bl',
    'mmse_slope', 'visit_number', 'MMSE', 'ADAS13',
    'mmse_variability', 'adas_mmse_discordance', 'mmse_acceleration'
]

# ============================================================================
# FEATURE ENGINEERING
# ============================================================================
def add_future_mmse_label(df, horizon=1.0, window=0.25):
    """Add MMSE score ~12 months later (no leakage: only looks forward)."""
    if "PTID" in df.columns:
        assert df["PTID"].nunique() == 1, "Data leakage risk: multiple patients in df!"
    future_mmse = []
    for i, row in df.iterrows():
        current_time = row["Years_bl"]
        target_time  = current_time + horizon
        mask = (
            (df["Years_bl"] >= target_time - window) &
            (df["Years_bl"] <= target_time + window)
        )
        candidates = df[mask]
        if len(candidates) > 0:
            future_mmse.append(
                candidates.iloc[
                    (candidates["Years_bl"] - target_time).abs().argmin()
                ]["MMSE"]
            )
        else:
            future_mmse.append(np.nan)
    df["MMSE_future12"] = future_mmse
    return df


def engineer_features(df):
    """Engineer temporal features using only past data at each time step."""
    df = df.sort_values("Years_bl").reset_index(drop=True)

    df["time_from_baseline"] = df["Years_bl"] - df["Years_bl"].iloc[0]
    df["age_bl"]             = df["AGE"].iloc[0]
    df["age_since_bl"]       = df["AGE"] - df["age_bl"]
    df["visit_number"]       = range(len(df))

    df["mmse_slope"]            = 0.0
    df["mmse_acceleration"]     = 0.0
    df["mmse_variability"]      = 0.0
    df["adas_mmse_discordance"] = 0.0

    for t in range(len(df)):
        df_past = df.iloc[:t + 1]

        if len(df_past) >= 2:
            dy = df_past["MMSE"].iloc[-1]   - df_past["MMSE"].iloc[-2]
            dt = df_past["Years_bl"].iloc[-1] - df_past["Years_bl"].iloc[-2]
            df.loc[t, "mmse_slope"] = dy / dt if dt != 0 else 0.0

        if t >= 2:
            df.loc[t, "mmse_acceleration"] = (
                df.loc[t, "mmse_slope"] - df.loc[t - 1, "mmse_slope"]
            )

        if len(df_past) >= 2:
            df.loc[t, "mmse_variability"] = df_past["MMSE"].std()

        df.loc[t, "adas_mmse_discordance"] = abs(
            df_past["ADAS13"].iloc[-1] - df_past["MMSE"].iloc[-1]
        )

    df = df.ffill().fillna(0)
    df = add_future_mmse_label(df)
    return df


# ============================================================================
# DATASET
# ============================================================================
class SequenceDataset(Dataset):
    def __init__(self, manifest, valid_patients, transform=None, max_seq_len=10):
        self.transform   = transform
        self.max_seq_len = max_seq_len
        self.sequences   = []
        self.scaler      = None

        manifest = manifest.copy()
        manifest["path"] = manifest["path"].str.replace("\\", "/", regex=False)
        manifest["path"] = "./AD_Multimodal/TFN_AD/" + manifest["path"]

        skipped_not_valid = 0
        skipped_errors    = 0
        processed         = 0

        for ptid in manifest["PTID"].unique():
            if ptid not in valid_patients:
                skipped_not_valid += 1
                continue

            try:
                patient_rows = manifest[manifest["PTID"] == ptid]
                if len(patient_rows) == 0:
                    continue

                df = pd.read_pickle(patient_rows.iloc[0]["path"])
                df = engineer_features(df)

                dx_seq = df["DX"].tolist()
                if "MCI" not in dx_seq:
                    continue

                mci_idx = dx_seq.index("MCI")
                ad_idx  = next(
                    (i for i, x in enumerate(dx_seq[mci_idx + 1:], start=mci_idx + 1)
                     if x in ["AD", "Dementia"]),
                    -1
                )

                if ad_idx != -1:
                    time_to_event = (
                        df["Years_bl"].iloc[ad_idx] - df["Years_bl"].iloc[mci_idx]
                    )
                    event = 1
                else:
                    time_to_event = (
                        df["Years_bl"].iloc[-1] - df["Years_bl"].iloc[mci_idx]
                    )
                    event = 0

                imgs, tabs, times, mmse_vals, mmse_future = [], [], [], [], []
                valid_visits = 0

                for _, visit in df.iterrows():
                    image_path = visit["image_path"].replace(
                        "/home/mason/ADNI_Dataset/",
                        "./AD_Multimodal/ADNI_Dataset/"
                    )
                    if not os.path.exists(image_path):
                        continue

                    img = Image.open(image_path).convert("RGB")
                    if self.transform:
                        img = self.transform(img)
                    imgs.append(img)

                    feature_cols = TEMPORAL_FEATURES + STATIC_FEATURES
                    row_vals = [
                        float(visit[col]) if col in visit.index else 0.0
                        for col in feature_cols
                    ]
                    tabs.append(np.array(row_vals, dtype=np.float32))

                    times.append(float(visit["Years_bl"]))
                    mmse_vals.append(float(visit["MMSE"]))
                    future_val = visit.get("MMSE_future12", np.nan)
                    mmse_future.append(float(future_val) if pd.notna(future_val) else np.nan)

                    valid_visits += 1
                    if valid_visits >= self.max_seq_len:
                        break

                if valid_visits < 2:
                    continue

                pad_len = self.max_seq_len - valid_visits
                for _ in range(pad_len):
                    imgs.append(torch.zeros_like(imgs[-1]))
                    tabs.append(np.zeros_like(tabs[-1]))
                    times.append(times[-1])
                    mmse_vals.append(np.nan)
                    mmse_future.append(np.nan)

                self.sequences.append({
                    "ptid":          str(ptid),
                    "imgs":          torch.stack(imgs),
                    "tabs":          np.array(tabs, dtype=np.float32),
                    "times":         np.array(times, dtype=np.float32),
                    "mmse":          np.array(mmse_vals, dtype=np.float32),
                    "mmse_future":   np.array(mmse_future, dtype=np.float32),
                    "seq_len":       valid_visits,
                    "time_to_event": float(time_to_event),
                    "event":         int(event),
                })
                processed += 1

            except Exception as e:
                skipped_errors += 1
                print(f"  ⚠️  Skipped {ptid}: {type(e).__name__}: {e}")
                continue

        print(f"  Processed                  : {processed}")
        print(f"  Skipped (not in valid set) : {skipped_not_valid}")
        print(f"  Skipped (errors)           : {skipped_errors}")

    def apply_scaler(self, scaler):
        self.scaler = scaler
        for seq in self.sequences:
            real_len = seq['seq_len']
            seq['tabs'][:real_len] = scaler.transform(seq['tabs'][:real_len])
            if real_len < len(seq['tabs']):
                seq['tabs'][real_len:] = 0.0

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        return (
            seq['imgs'], seq['tabs'], seq['times'], seq['mmse'],
            seq['mmse_future'], seq['seq_len'],
            seq['time_to_event'], seq['event'], seq['ptid'],
        )


# ============================================================================
# MODEL
# ============================================================================
class TensorFusion(nn.Module):
    def __init__(self, v_dim, d_dim, t_dim, dropout=0.1):
        super().__init__()
        # +1 on each dim because TensorFusion appends a bias-1 to each vector
        self.output_dim = (v_dim + 1) * (d_dim + 1) * (t_dim + 1)
        self.dropout    = nn.Dropout(dropout)

    def forward(self, v, d, t):
        bs = v.shape[0]
        v1 = torch.cat([v, torch.ones(bs, 1, device=v.device)], dim=1)
        d1 = torch.cat([d, torch.ones(bs, 1, device=d.device)], dim=1)
        t1 = torch.cat([t, torch.ones(bs, 1, device=t.device)], dim=1)
        fusion = torch.einsum('bi,bj,bk->bijk', v1, d1, t1).view(bs, -1)
        return fusion   # dropout not applied here (applied via weight_decay)


class AttentionImageEncoder(nn.Module):
    def __init__(self, out_dim=256):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        for param in list(base.parameters())[:-20]:
            param.requires_grad = False
        self.features    = nn.Sequential(*list(base.children())[:-2])
        self.attention   = nn.Sequential(
            nn.Conv2d(512, 256, 1), nn.BatchNorm2d(256), nn.GELU(),
            nn.Conv2d(256, 1,   1), nn.Sigmoid()
        )
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.proj        = nn.Linear(512, out_dim)

    def forward(self, x):
        feats = self.features(x)
        feats = feats * self.attention(feats)
        return self.proj(self.global_pool(feats).view(x.size(0), -1))


class SupervisedTemporalFusionAutoencoder(nn.Module):
    def __init__(self, tab_dim, config):
        super().__init__()
        self.config = config

        self.img_encoder  = AttentionImageEncoder(out_dim=config['img_out_dim'])
        self.tab_encoder  = nn.Sequential(
            nn.Linear(tab_dim, 128), nn.LayerNorm(128), nn.GELU(),
            nn.Dropout(config['dropout']),
            nn.Linear(128, config['tab_out_dim']), nn.GELU()
        )
        self.time_encoder = nn.Sequential(nn.Linear(1, 16), nn.GELU())

        self.fusion = TensorFusion(
            v_dim=config['img_out_dim'],
            d_dim=config['tab_out_dim'],
            t_dim=16,
            dropout=config['dropout']
        )

        # Fix A: time_encoder outputs 16-dim; after appending bias-1 in
        # TensorFusion it becomes 17-dim. Correct multiplier is 17, not 16.
        # (img_out_dim+1)=257, (tab_out_dim+1)=65, (16+1)=17 → 257*65*17=283,985
        fusion_dim = (config['img_out_dim'] + 1) * (config['tab_out_dim'] + 1) * 17

        self.fusion_proj = nn.Sequential(
            nn.Linear(fusion_dim, config['latent_dim']),
            nn.BatchNorm1d(config['latent_dim']), nn.GELU()
        )

        # Fix B: bidirectional=True, matching all ablation models.
        # temporal_proj must take lstm_hidden*2 as input.
        self.lstm = nn.LSTM(
            input_size=config['latent_dim'],
            hidden_size=config['lstm_hidden'],
            num_layers=config['lstm_layers'],
            batch_first=True,
            dropout=config['dropout'] if config['lstm_layers'] > 1 else 0,
            bidirectional=True           # Fix B
        )
        self.temporal_proj = nn.Sequential(
            nn.Linear(config['lstm_hidden'] * 2, config['latent_dim']), nn.GELU()
        )

        self.survival_head = nn.Sequential(
            nn.Linear(config['latent_dim'], 64), nn.GELU(),
            nn.Dropout(config['dropout']),
            nn.Linear(64, 32), nn.GELU(),
            nn.Linear(32, 1)
        )
        self.mmse_head = nn.Sequential(
            nn.Linear(config['latent_dim'], 32), nn.GELU(),
            nn.Linear(32, 1)
        )

    def encode_visit(self, img, tab, time):
        v = self.img_encoder(img)
        d = self.tab_encoder(tab)
        t = self.time_encoder(time.unsqueeze(1))
        v = F.normalize(v, dim=1)
        d = F.normalize(d, dim=1)
        t = F.normalize(t, dim=1)
        z = self.fusion(v, d, t)
        return self.fusion_proj(z.view(z.size(0), -1))

    def forward(self, img_seq, tab_seq, time_seq, seq_lengths):
        bs, seq_len = img_seq.shape[:2]

        z_list = [
            self.encode_visit(img_seq[:, t], tab_seq[:, t], time_seq[:, t])
            for t in range(seq_len)
        ]
        z_seq = torch.stack(z_list, dim=1)  # (bs, seq_len, latent_dim)

        packed = nn.utils.rnn.pack_padded_sequence(
            z_seq, seq_lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (h_n, _) = self.lstm(packed)

        # Fix B: BiLSTM final state — concatenate last forward and backward layers.
        # h_n shape: (num_layers*2, bs, lstm_hidden).
        # h_n[-2] = last layer forward, h_n[-1] = last layer backward.
        h_final = torch.cat([h_n[-2], h_n[-1]], dim=1)   # (bs, lstm_hidden*2)
        z_final = self.temporal_proj(h_final)             # (bs, latent_dim)

        # Fix C: consistent name — mmse_pred (singular) used everywhere.
        mmse_pred = torch.stack(
            [self.mmse_head(z_seq[:, t]) for t in range(seq_len)],
            dim=1
        )   # (bs, seq_len, 1)

        return z_final, self.survival_head(z_final), mmse_pred


def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)


# ============================================================================
# LOSS
# ============================================================================
def cox_partial_likelihood_loss_efron(risk_scores, times, events):
    device = risk_scores.device
    mask   = torch.isfinite(risk_scores) & torch.isfinite(times)
    risk_scores, times, events = risk_scores[mask], times[mask], events[mask]

    if events.sum() == 0:
        return torch.tensor(0.0, device=device)

    order       = torch.argsort(times)
    risk_scores = risk_scores[order]
    times       = times[order]
    events      = events[order]

    log_risk  = risk_scores.view(-1)
    hazard    = torch.exp(log_risk)
    loss      = torch.tensor(0.0, device=device)

    for t in torch.unique(times[events == 1]):
        at_risk   = times >= t
        died      = (times == t) & (events == 1)
        if died.sum() == 0:
            continue
        n_died    = died.sum().float()
        risk_set  = hazard[at_risk].sum()
        tied_risk = hazard[died].sum()
        for j in range(int(n_died)):
            loss += torch.log(risk_set - (j / n_died) * tied_risk + 1e-7)
        loss -= log_risk[died].sum()

    return loss / (events.sum() + 1e-7)


# ============================================================================
# TRAINING & VALIDATION
# ============================================================================
def train_epoch(model, loader, optimizer, config, device):
    model.train()
    total_loss = total_cox = total_mmse = 0.0

    for batch in loader:
        imgs, tabs, times, mmse, mmse_future, seq_lens, t_event, event, _ = batch

        imgs          = imgs.to(device)
        tabs          = torch.FloatTensor(tabs).to(device)
        times         = torch.FloatTensor(times).to(device)
        mmse_future_v = torch.FloatTensor(mmse_future).to(device)
        seq_lens      = torch.LongTensor(seq_lens)
        t_event       = t_event.float().to(device)
        event         = event.float().to(device)

        # Fix C+D: forward returns mmse_pred; iterate over mmse_pred.shape[1]
        z_final, risk_scores, mmse_pred = model(imgs, tabs, times, seq_lens)

        loss_cox = cox_partial_likelihood_loss_efron(
            risk_scores.squeeze(), t_event, event
        )

        loss_mmse = torch.tensor(0.0, device=device)
        count = 0
        for t in range(mmse_pred.shape[1]):
            valid_mask = ~torch.isnan(mmse_future_v[:, t])
            if valid_mask.any():
                loss_mmse = loss_mmse + F.mse_loss(
                    mmse_pred[:, t].squeeze()[valid_mask],
                    mmse_future_v[:, t][valid_mask]
                )
                count += 1
        if count > 0:
            loss_mmse = loss_mmse / count

        # Fix E: L1 penalty re-enabled
        l1_survival = sum(p.abs().sum() for p in model.survival_head.parameters())

        loss = (
            config['alpha_survival'] * loss_cox
            + config['alpha_mmse']   * loss_mmse
            + 1e-4                   * l1_survival
        )

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        total_cox  += loss_cox.item()
        total_mmse += loss_mmse.item()

    n = len(loader)
    return {'total': total_loss / n, 'cox': total_cox / n, 'mmse': total_mmse / n}


def validate(model, loader, device):
    model.eval()
    all_risks, all_times, all_events = [], [], []

    with torch.no_grad():
        for batch in loader:
            imgs, tabs, times, _, _, seq_lens, t_event, event, _ = batch
            imgs     = imgs.to(device)
            tabs     = torch.FloatTensor(tabs).to(device)
            times    = torch.FloatTensor(times).to(device)
            seq_lens = torch.LongTensor(seq_lens)

            _, risk_scores, _ = model(imgs, tabs, times, seq_lens)
            all_risks.extend(risk_scores.cpu().numpy().flatten())
            all_times.extend(t_event.numpy())
            all_events.extend(event.numpy())

    all_events = np.array(all_events).astype(bool)
    c_index    = concordance_index(
        np.array(all_times), -np.array(all_risks), all_events
    )
    return c_index, np.array(all_risks), np.array(all_times), all_events


# ============================================================================
# EXPORT
# ============================================================================
def export_latent_features(model, loader, device, output_path):
    model.eval()
    rows = []
    BASELINE_FEATURES = ['AGE', 'PTGENDER', 'PTEDUCAT', 'ADAS13']
    tab_feature_names = TEMPORAL_FEATURES + STATIC_FEATURES

    with torch.no_grad():
        for batch in loader:
            imgs, tabs, times, mmse, mmse_future, seq_lens, t_event, event, ptids = batch
            imgs     = imgs.to(device)
            tabs     = tabs.to(device)
            times    = times.to(device)
            seq_lens = seq_lens.cpu().long()

            for i in range(len(ptids)):
                slen = seq_lens[i].item()
                for t in range(slen):
                    z_visit = model.encode_visit(
                        imgs[i:i+1, t], tabs[i:i+1, t], times[i:i+1, t]
                    )
                    base_dataset = (
                        loader.dataset.dataset
                        if hasattr(loader.dataset, 'dataset')
                        else loader.dataset
                    )
                    tab_vals_unscaled = base_dataset.scaler.inverse_transform(
                        tabs[i, t].cpu().numpy().reshape(1, -1)
                    )[0]

                    row = {
                        "PTID":          str(ptids[i]),
                        "Years_bl":      float(times[i, t].cpu()),
                        "MMSE":          float(mmse[i, t]),
                        "time_to_event": float(t_event[i]),
                        "event":         int(event[i]),
                    }
                    for feat in BASELINE_FEATURES:
                        feat_encoded = feat + '_encoded' if feat == 'PTGENDER' else feat
                        if feat_encoded in tab_feature_names:
                            idx_f     = tab_feature_names.index(feat_encoded)
                            row[feat] = float(tab_vals_unscaled[idx_f])
                    for k, val in enumerate(z_visit[0].cpu().numpy()):
                        row[f"z_{k}"] = float(val)
                    rows.append(row)

    df = pd.DataFrame(rows).sort_values(["PTID", "Years_bl"])
    df.to_csv(output_path, index=False)
    print(f"\n✓ Exported to {output_path}")
    print(f"  Patients        : {df['PTID'].nunique()}")
    print(f"  Visits          : {len(df)}")
    print(f"  Latent features : {len([c for c in df.columns if c.startswith('z_')])}")
    return df


def export_patient_level_latent(model, loader, device, output_path):
    model.eval()
    rows = []

    with torch.no_grad():
        for batch in loader:
            imgs, tabs, times, mmse, mmse_future, seq_lens, t_event, event, ptids = batch

            imgs       = imgs.to(device)
            tabs       = tabs.to(device)
            times_dev  = times.to(device)
            seq_lens_t = torch.LongTensor(seq_lens)

            z_final, risk_scores, _ = model(imgs, tabs, times_dev, seq_lens_t)

            bs, seq_len = imgs.shape[:2]
            z_visits = []
            for t in range(seq_len):
                z_t = model.encode_visit(imgs[:, t], tabs[:, t], times_dev[:, t])
                z_visits.append(z_t.cpu().numpy())
            z_visits = np.array(z_visits)   # (seq_len, bs, latent_dim)

            for i in range(len(ptids)):
                slen = int(seq_lens[i])
                row  = {
                    "PTID":          str(ptids[i]),
                    "time_to_event": float(t_event[i]),
                    "event":         int(event[i]),
                    "risk_score":    float(risk_scores[i].cpu()),
                    "MMSE":          float(mmse[i, slen - 1]),
                    "Years_bl":      float(times[i, slen - 1]),
                    "n_visits":      slen,
                }
                for k, val in enumerate(z_final[i].cpu().numpy()):
                    row[f"z_final_{k}"] = float(val)

                z_last_visit = z_visits[slen - 1, i, :]
                for k, val in enumerate(z_last_visit):
                    row[f"z_last_{k}"] = float(val)

                real_z = z_visits[:slen, i, :]
                if slen >= 2:
                    visit_times = times[i, :slen].numpy()
                    dt          = visit_times - visit_times[0]
                    if dt[-1] > 0:
                        for k in range(real_z.shape[1]):
                            slope = (
                                np.cov(dt, real_z[:, k])[0, 1]
                                / (np.var(dt) + 1e-8)
                            )
                            row[f"z_slope_{k}"] = float(slope)
                    else:
                        for k in range(real_z.shape[1]):
                            row[f"z_slope_{k}"] = 0.0
                else:
                    for k in range(real_z.shape[1]):
                        row[f"z_slope_{k}"] = 0.0

                rows.append(row)

    df = pd.DataFrame(rows).sort_values("PTID").reset_index(drop=True)
    df.to_csv(output_path, index=False)
    n_zf = len([c for c in df.columns if c.startswith("z_final_")])
    n_zs = len([c for c in df.columns if c.startswith("z_slope_")])
    print(f"\n✓ Patient-level export → {output_path}")
    print(f"  Patients      : {len(df)}")
    print(f"  z_final feats : {n_zf}")
    print(f"  z_slope feats : {n_zs}")
    print(f"  Events        : {df['event'].sum()} ({100*df['event'].mean():.1f}%)")
    return df


# ============================================================================
# MAIN
# ============================================================================
def main():
    print("=" * 80)
    print("ENHANCED MCI-TO-AD PREDICTION  (all shape bugs fixed)")
    print("=" * 80)

    device = CONFIG['device']
    print(f"\nDevice: {device}")

    # Verify fusion_dim at startup — catches any future dimension drift early
    _fd = (CONFIG['img_out_dim'] + 1) * (CONFIG['tab_out_dim'] + 1) * 17
    print(f"Expected fusion_dim : {_fd}  "
          f"({CONFIG['img_out_dim']+1} × {CONFIG['tab_out_dim']+1} × 17)")

    # ------------------------------------------------------------------
    # Load manifest and valid patients
    # ------------------------------------------------------------------
    print("\nLoading data...")
    manifest = pd.read_csv("./AD_Multimodal/TFN_AD/AD_Patient_Manifest.csv")

    with open('VALID_PATIENTS.pkl', 'rb') as f:
        VALID_PATIENTS = pickle.load(f)
    VALID_PATIENTS   = set(str(p) for p in VALID_PATIENTS)
    manifest["PTID"] = manifest["PTID"].astype(str)

    print(f"Valid patients in pkl : {len(VALID_PATIENTS)}")
    overlap = set(manifest["PTID"].unique()) & VALID_PATIENTS
    print(f"Overlap with manifest : {len(overlap)}")
    if len(overlap) == 0:
        raise RuntimeError("No PTID overlap — check VALID_PATIENTS.pkl types.")

    # ------------------------------------------------------------------
    # Transforms & Dataset
    # ------------------------------------------------------------------
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    print("\nCreating dataset...")
    dataset = SequenceDataset(manifest, VALID_PATIENTS, transform,
                              max_seq_len=CONFIG['max_seq_len'])
    print(f"\nTotal sequences: {len(dataset)}")
    if len(dataset) == 0:
        raise RuntimeError("Dataset is empty after processing.")

    # ------------------------------------------------------------------
    # Train / val split
    # ------------------------------------------------------------------
    # split_df         = pd.read_csv("patient_splits.csv")
    # split_df["PTID"] = split_df["PTID"].astype(str)
    # train_ptids = set(split_df[split_df["split"] == "train"]["PTID"].tolist())
    # val_ptids   = set(split_df[split_df["split"] == "val"]["PTID"].tolist())

    # train_indices = [i for i, s in enumerate(dataset.sequences)
    #                  if str(s['ptid']) in train_ptids]
    # val_indices   = [i for i, s in enumerate(dataset.sequences)
    #                  if str(s['ptid']) in val_ptids]

    with open('TRAIN_PATIENTS.pkl', 'rb') as f:
        TRAIN_PATIENTS = pickle.load(f)
    with open('VAL_PATIENTS.pkl', 'rb') as f:
        VAL_PATIENTS = pickle.load(f)

    train_indices = [i for i, s in enumerate(dataset.sequences) if s['ptid'] in TRAIN_PATIENTS]
    val_indices   = [i for i, s in enumerate(dataset.sequences) if s['ptid'] in VAL_PATIENTS]

    if len(train_indices) == 0:
        raise RuntimeError("No training patients matched patient_splits.csv.")
    if len(val_indices) == 0:
        raise RuntimeError("No val patients matched patient_splits.csv.")

    train_dataset = torch.utils.data.Subset(dataset, train_indices)
    val_dataset   = torch.utils.data.Subset(dataset, val_indices)
    print(f"Train: {len(train_indices)}  |  Val: {len(val_indices)}")

    # ------------------------------------------------------------------
    # Scaler — fit on train only
    # ------------------------------------------------------------------
    train_tabs = np.vstack([
        dataset.sequences[i]['tabs'][:dataset.sequences[i]['seq_len']]
        for i in train_indices
    ])
    scaler = StandardScaler()
    scaler.fit(train_tabs)
    dataset.apply_scaler(scaler)
    print("Scaler fitted on train only ✓")

    # ------------------------------------------------------------------
    # DataLoaders
    # ------------------------------------------------------------------
    train_loader        = DataLoader(train_dataset, batch_size=CONFIG['batch_size'],
                                     shuffle=True,  num_workers=0)
    val_loader          = DataLoader(val_dataset,   batch_size=CONFIG['batch_size'],
                                     shuffle=False, num_workers=0)
    train_loader_export = DataLoader(train_dataset, batch_size=CONFIG['batch_size'],
                                     shuffle=False, num_workers=0)

    # ------------------------------------------------------------------
    # Model
    # ------------------------------------------------------------------
    tab_dim = dataset.sequences[0]['tabs'].shape[1]
    print(f"\nTab feature dimension : {tab_dim}")
    model = SupervisedTemporalFusionAutoencoder(tab_dim, CONFIG).to(device)
    model.apply(init_weights)
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

    # Sanity check — forward pass on a tiny fake batch before training starts
    print("\nRunning shape sanity check...")
    with torch.no_grad():
        _imgs  = torch.zeros(2, 2, 3, 224, 224).to(device)
        _tabs  = torch.zeros(2, 2, tab_dim).to(device)
        _times = torch.zeros(2, 2).to(device)
        _lens  = torch.LongTensor([2, 2])
        _zf, _rs, _mp = model(_imgs, _tabs, _times, _lens)
        print(f"  z_final : {tuple(_zf.shape)}  ✓")
        print(f"  risk    : {tuple(_rs.shape)}  ✓")
        print(f"  mmse    : {tuple(_mp.shape)}  ✓")
    del _imgs, _tabs, _times, _lens, _zf, _rs, _mp

    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=CONFIG['lr'],
                                  weight_decay=CONFIG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5
    )

    # ------------------------------------------------------------------
    # Training
    # ------------------------------------------------------------------
    print("\n" + "=" * 80)
    print("TRAINING")
    print("=" * 80)

    best_c_index = 0.0
    patience_ctr = 0

    for epoch in range(CONFIG['epochs']):
        train_losses = train_epoch(model, train_loader, optimizer, CONFIG, device)
        val_c_index, _, _, _ = validate(model, val_loader, device)
        scheduler.step(val_c_index)

        print(f"\nEpoch {epoch + 1}/{CONFIG['epochs']}")
        print(f"  Loss : {train_losses['total']:.4f}  "
              f"(Cox: {train_losses['cox']:.4f}  MMSE: {train_losses['mmse']:.4f})")
        print(f"  Val C-index : {val_c_index:.4f}")

        if val_c_index > best_c_index:
            best_c_index = val_c_index
            torch.save({
                'epoch':            epoch,
                'model_state_dict': model.state_dict(),
                'c_index':          best_c_index,
            }, 'best_model_v4.pth')
            print(f"  ✓ New best: {best_c_index:.4f}")
            patience_ctr = 0
        else:
            patience_ctr += 1

        if patience_ctr >= 15:
            print("\n⚠  Early stopping.")
            break

    # ------------------------------------------------------------------
    # Export — train and val through separate loaders (no leakage)
    # ------------------------------------------------------------------
    print("\n" + "=" * 80)
    print("EXPORTING")
    print("=" * 80)

    checkpoint = torch.load('best_model_v4.pth', weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"\nLoaded best model (epoch {checkpoint['epoch'] + 1}, "
          f"C-index {checkpoint['c_index']:.4f})")

        # Replace the separate train/val export with this:
    full_loader_export = DataLoader(
        torch.utils.data.Subset(dataset, list(range(len(dataset)))),
        batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0
    )
    
    patient_combined = export_patient_level_latent(
        model, full_loader_export, device, "latent_patient_level_V5.csv"
    )
    patient_combined['split'] = patient_combined['PTID'].apply(
        lambda p: 'train' if str(p) in TRAIN_PATIENTS else 'val'
    )
    patient_combined.to_csv("latent_patient_level_V5.csv", index=False)
    
    latent_combined = export_latent_features(
        model, full_loader_export, device, "latent_improved_autoencoder_V5.csv"
    )
    latent_combined['split'] = latent_combined['PTID'].apply(
        lambda p: 'train' if str(p) in TRAIN_PATIENTS else 'val'
    )
    latent_combined.to_csv("latent_improved_autoencoder_V5.csv", index=False)

    print("\n" + "=" * 80)
    print("✓ COMPLETE")
    print("=" * 80)
    print(f"\nBest val C-index : {best_c_index:.4f}")
    print("Ready for JMBayes2 in R!")


if __name__ == "__main__":
    main()

ENHANCED MCI-TO-AD PREDICTION  (all shape bugs fixed)

Device: cuda
Expected fusion_dim : 283985  (257 × 65 × 17)

Loading data...
Valid patients in pkl : 382
Overlap with manifest : 382

Creating dataset...
  Processed                  : 161
  Skipped (not in valid set) : 0
  Skipped (errors)           : 0

Total sequences: 161
Train: 128  |  Val: 33
Scaler fitted on train only ✓

Tab feature dimension : 16
Parameters: 48,508,003

Running shape sanity check...
  z_final : (2, 128)  ✓
  risk    : (2, 1)  ✓
  mmse    : (2, 2, 1)  ✓

TRAINING

Epoch 1/100
  Loss : 96.4505  (Cox: 2.2917  MMSE: 629.3533)
  Val C-index : 0.6699
  ✓ New best: 0.6699

Epoch 2/100
  Loss : 83.2307  (Cox: 2.2663  MMSE: 541.3694)
  Val C-index : 0.6699

Epoch 3/100
  Loss : 76.0744  (Cox: 2.1478  MMSE: 494.3349)
  Val C-index : 0.6990
  ✓ New best: 0.6990

Epoch 4/100
  Loss : 68.8879  (Cox: 2.0924  MMSE: 446.7407)
  Val C-index : 0.7346
  ✓ New best: 0.7346

Epoch 5/100
  Loss : 67.7311  (Cox: 2.0474  MMSE: 439

Train C-index: 0.8035
Val C-index:   0.8350


In [10]:
import pandas as pd
import pickle

manifest = pd.read_csv("./AD_Multimodal/TFN_AD/AD_Patient_Manifest.csv")
manifest["PTID"] = manifest["PTID"].astype(str)
manifest["path"] = manifest["path"].str.replace("\\", "/", regex=False)
manifest["path"] = "./AD_Multimodal/TFN_AD/" + manifest["path"]

with open('VALID_PATIENTS.pkl', 'rb') as f:
    VALID_PATIENTS = pickle.load(f)
VALID_PATIENTS = set(str(p) for p in VALID_PATIENTS)

follow_ups = []
time_to_events = []
events = []

for ptid in VALID_PATIENTS:
    rows = manifest[manifest["PTID"] == ptid]
    if rows.empty:
        continue
    try:
        df = pd.read_pickle(rows.iloc[0]["path"])
        dx_seq = df["DX"].tolist()
        if "MCI" not in dx_seq:
            continue
        
        mci_idx = dx_seq.index("MCI")
        max_years = df["Years_bl"].max()
        mci_years = df["Years_bl"].iloc[mci_idx]
        follow_up = max_years - mci_years
        follow_ups.append(follow_up)
        
        ad_idx = next(
            (i for i, x in enumerate(dx_seq[mci_idx+1:], start=mci_idx+1)
             if x in ["AD", "Dementia"]), -1
        )
        if ad_idx != -1:
            tte = df["Years_bl"].iloc[ad_idx] - mci_years
            time_to_events.append(tte)
            events.append(1)
        else:
            time_to_events.append(follow_up)
            events.append(0)
    except:
        pass

fu = pd.Series(follow_ups)
tte = pd.Series(time_to_events)
ev = pd.Series(events)

print(f"Patients: {len(fu)}")
print(f"Event rate: {ev.mean():.1%}")
print(f"\nFollow-up distribution (years from MCI diagnosis):")
print(fu.describe().round(2))
print(f"\nPatients with >3yr follow-up: {(fu > 3).sum()}")
print(f"Patients with >5yr follow-up: {(fu > 5).sum()}")
print(f"\nTime-to-event distribution:")
print(tte.describe().round(2))

Patients: 161
Event rate: 46.0%

Follow-up distribution (years from MCI diagnosis):
count    161.00
mean       2.84
std        0.65
min        0.00
25%        2.98
50%        3.01
75%        3.03
max        3.46
dtype: float64

Patients with >3yr follow-up: 100
Patients with >5yr follow-up: 0

Time-to-event distribution:
count    161.00
mean       2.26
std        0.97
min        0.00
25%        1.49
50%        2.98
75%        3.01
max        3.46
dtype: float64


In [1]:
import pandas as pd
import pickle

# Load manifest
manifest = pd.read_csv("./AD_Multimodal/TFN_AD/AD_Patient_Manifest.csv")
manifest["PTID"] = manifest["PTID"].astype(str)
manifest["path"] = manifest["path"].str.replace("\\", "/", regex=False)
manifest["path"] = "./AD_Multimodal/TFN_AD/" + manifest["path"]

# Load valid patients
with open('VALID_PATIENTS.pkl', 'rb') as f:
    VALID_PATIENTS = pickle.load(f)
VALID_PATIENTS = set(str(p) for p in VALID_PATIENTS)

# Storage
follow_ups = []
time_to_events = []
events = []
visit_counts = []

# Loop through patients
for ptid in VALID_PATIENTS:
    rows = manifest[manifest["PTID"] == ptid]
    if rows.empty:
        continue

    try:
        df = pd.read_pickle(rows.iloc[0]["path"])
        
        # Count visits (total rows = total visits)
        num_visits = len(df)

        dx_seq = df["DX"].tolist()
        if "MCI" not in dx_seq:
            continue

        # MCI index and time
        mci_idx = dx_seq.index("MCI")
        mci_years = df["Years_bl"].iloc[mci_idx]

        # Follow-up = last visit - MCI visit
        max_years = df["Years_bl"].max()
        follow_up = max_years - mci_years

        # Find conversion (AD/Dementia after MCI)
        ad_idx = next(
            (i for i, x in enumerate(dx_seq[mci_idx+1:], start=mci_idx+1)
             if x in ["AD", "Dementia"]),
            -1
        )

        if ad_idx != -1:
            tte = df["Years_bl"].iloc[ad_idx] - mci_years
            event = 1
        else:
            tte = follow_up
            event = 0

        # Append ONLY after all filters (ensures consistency)
        follow_ups.append(follow_up)
        time_to_events.append(tte)
        events.append(event)
        visit_counts.append(num_visits)

    except Exception as e:
        continue

# Convert to Series
fu = pd.Series(follow_ups)
tte = pd.Series(time_to_events)
ev = pd.Series(events)
vc = pd.Series(visit_counts)

# Print results
print(f"Patients: {len(fu)}")
print(f"Event rate: {ev.mean():.1%}")

print(f"\nVisits per patient:")
print(vc.describe().round(2))
print(f"Mean visits per patient: {vc.mean():.2f}")

print(f"\nFollow-up distribution (years from MCI diagnosis):")
print(fu.describe().round(2))
print(f"Patients with >3yr follow-up: {(fu > 3).sum()}")
print(f"Patients with >5yr follow-up: {(fu > 5).sum()}")

print(f"\nTime-to-event distribution:")
print(tte.describe().round(2))

Patients: 161
Event rate: 46.0%

Visits per patient:
count    161.00
mean       5.89
std        0.37
min        4.00
25%        6.00
50%        6.00
75%        6.00
max        6.00
dtype: float64
Mean visits per patient: 5.89

Follow-up distribution (years from MCI diagnosis):
count    161.00
mean       2.84
std        0.65
min        0.00
25%        2.98
50%        3.01
75%        3.03
max        3.46
dtype: float64
Patients with >3yr follow-up: 100
Patients with >5yr follow-up: 0

Time-to-event distribution:
count    161.00
mean       2.26
std        0.97
min        0.00
25%        1.49
50%        2.98
75%        3.01
max        3.46
dtype: float64


ENHANCED MCI-TO-AD PREDICTION

Device: cuda

Loading data...

Loading valid patient list...
Valid patients in pkl : 161
Overlap with manifest : 161

Creating dataset...
  Processed : 161 valid patients
  Skipped (not in valid set) : 221
  Skipped (errors)           : 0

Total sequences: 161
Train: 128  |  Val (test): 33
Scaler fitted on train split only — no val leakage ✓

Tab feature dimension : 16
Initializing model...
Parameters: 48,508,003

TRAINING

Epoch 1/100
  Loss : 1.9467  (Cox: 2.2903  MMSE: 0.0000)
  Val C-index : 0.7605
  ✓ New best: 0.7605

Epoch 2/100
  Loss : 1.8838  (Cox: 2.2162  MMSE: 0.0000)
  Val C-index : 0.7896
  ✓ New best: 0.7896

Epoch 3/100
  Loss : 1.8452  (Cox: 2.1709  MMSE: 0.0000)
  Val C-index : 0.7573

Epoch 4/100
  Loss : 1.5444  (Cox: 1.8170  MMSE: 0.0000)
  Val C-index : 0.7314

Epoch 5/100
  Loss : 1.6537  (Cox: 1.9456  MMSE: 0.0000)
  Val C-index : 0.7443

Epoch 6/100
  Loss : 1.4405  (Cox: 1.6947  MMSE: 0.0000)
  Val C-index : 0.7638

Epoch 7/100
 

In [14]:
train_c, _, _, _ = validate(model, train_loader, device)
val_c, _, _, _ = validate(model, val_loader, device)
print(f"Train C-index: {train_c:.4f}")
print(f"Val C-index:   {val_c:.4f}")

Train C-index: 0.8035
Val C-index:   0.8350


In [6]:
print(f"train_ptids size: {len(train_ptids)}")
print(f"val_ptids size:   {len(val_ptids)}")
print(f"Sample train IDs: {list(train_ptids)[:5]}")

NameError: name 'train_ptids' is not defined